# step1_Ecearth3historical_downscaling_F1 - Calibration of F1 (global)

**F1 = analogue-based downscaling** (your supervisor's method), learned between:

- **EC-Earth3 historical** at its native resolution (~0.7 deg), as input;
- **ERA5 0.25 deg**, as target.

No regridding, no WDF, no QDM: the analogue method does **not** require day-by-day consistency between EC-Earth3 and ERA5
(the two datasets are supplied separately, not paired in time).

**The method is not written here**: this notebook prepares the data and calls `train_f1` from `f1_downscaler.py`
(to be filled in with the supervisor's method).

F1 is then applied to all GCM runs (notebook step2), which gives the "ERA5-like" GCMs at 0.25 deg.

**Variables and periods**: `pr`, `tasmax`, `tasmin` over `TRAIN_PERIOD` (1981-2014: F1 trained on the whole
period for now; a split, e.g. 1981-2003 / 2004-2014, can be done later for an out-of-sample validation), and
`rsds` (solar radiation) over the part of `TRAIN_PERIOD` from 2005 on, ERA5 radiation starting in 2005.

Whole globe (no region box). Volume: global ERA5 at 0.25 deg is ~1.5 GB per variable and per year (float32, uncompressed): the method will need to process the data in blocks (years, or spatial tiles).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py, f1_downscaler.py: this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"] + cfg.get("extra_variables", [])   # pr, tasmax, tasmin (+ rsds)

TRAIN_PERIOD = ("1981", "2014")     # whole period for now (EC-Earth3 historical ends in 2014); later e.g. ("1981", "2003")
RSDS_FIRST_YEAR = "2005"            # ERA5 radiation starts in 2005: shorter training period for rsds

def period_of(v):
    """Training period of a variable: TRAIN_PERIOD, starting in 2005 for rsds."""
    return (max(TRAIN_PERIOD[0], RSDS_FIRST_YEAR), TRAIN_PERIOD[1]) if v == "rsds" else TRAIN_PERIOD

import f1_downscaler as f1          # <- file to complete (supervisor's method)
F1_DIR = bu.out_dir(cfg, "plan_B", "F1_downscaling", f"{TRAIN_PERIOD[0]}-{TRAIN_PERIOD[1]}")
print("F1 folder:", F1_DIR)

## 1. Training data (not paired in time)

In [ ]:
VARS_TRAIN = [v for v in VARS if bu.dataset_available(cfg, "gcm_historical", v) and bu.dataset_available(cfg, "era5", v)]
for v in VARS:
    if v not in VARS_TRAIN:
        print(f"[skipped] {v}: missing for EC-Earth3 historical or ERA5 (notebook 00)")
gcm_hist = {v: bu.subset_time(bu.read_raw(cfg, "gcm_historical", v), *period_of(v)) for v in VARS_TRAIN}
era5 = {v: bu.subset_time(bu.read_raw(cfg, "era5", v), *period_of(v)) for v in VARS_TRAIN}
for v in VARS_TRAIN:
    g, e = gcm_hist[v], era5[v]
    print(f"{v:7s} {period_of(v)[0]}-{period_of(v)[1]} | native GCM: {g.sizes['lat']}x{g.sizes['lon']} "
          f"(step ~{abs(float(np.diff(g.lat).mean())):.2f} deg, {g.sizes['time']} days)"
          f" | ERA5: {e.sizes['lat']}x{e.sizes['lon']} (step ~{abs(float(np.diff(e.lat).mean())):.2f} deg, {e.sizes['time']} days)")

## 2. Training (supervisor's method: to be written in `f1_downscaler.train_f1`)

In [ ]:
try:
    f1.train_f1(gcm_hist, era5, F1_DIR)
    import json
    (F1_DIR / "f1_training_info.json").write_text(json.dumps(
        {"variables": VARS_TRAIN, "periods": {v: list(period_of(v)) for v in VARS_TRAIN}}, indent=2))
    print("F1 trained ->", F1_DIR)
except NotImplementedError as e:
    print("PLACEHOLDER:", e)

## 3. Notes

- Contract expected by notebook step2: `f1_downscaler.apply_f1(gcm_sim_dict, F1_DIR, target_lat, target_lon) -> dict` of 0.25 deg DataArrays (see `f1_downscaler.py`).
- Variables: `pr`, `tasmax`, `tasmin` (check `tasmin < tasmax` and `pr >= 0` after downscaling), and `rsds` (>= 0)
  when available; `rsds` is trained on a shorter period (from 2005). Variables and periods are written to
  `F1_DIR/f1_training_info.json`.
- Out-of-sample test (later): rerun with `TRAIN_PERIOD = ("1981", "2003")`, then step2 with the same period, and the
  validation notebook with `F1_TRAIN_PERIOD = ("1981", "2003")`, `EVAL_PERIOD = ("2004", "2014")`. Outputs go to
  separate folders (`F1_<period>`), nothing is overwritten.